# MQTT Subscriber

## Import the required module

In [ ]:
import paho.mqtt.client as mqtt
import json
import redis

## Create a new MQTT client

In [ ]:
client = mqtt.Client()

## Configure REDIS

In [ ]:
# Configure here your Redis connection parameters
REDIS_HOST = "redis-15448.c300.eu-central-1-1.ec2.cloud.redislabs.com"
REDIS_PORT = "15448"
REDIS_USER = "default"
REDIS_PASS = "QVmVhSJy9xbmT9qtySmpMFCgWQKx0otn"

def get_redis_client_instance(mac_address: str):
	redis_client = redis.Redis(
		host=REDIS_HOST,
		port=REDIS_PORT,
		username=REDIS_USER,
		password=REDIS_PASS,
	)
	temp_timeseries_name = f"{mac_address}:temperature"
	humidity_timeseries_name = f"{mac_address}:humidity"
 
	try:
		redis_client.ts().create(temp_timeseries_name)
	except redis.ResponseError:
		pass

	try:
		redis_client.ts().create(humidity_timeseries_name)
	except redis.ResponseError:
		pass

	return redis_client, temp_timeseries_name, humidity_timeseries_name


## Define the callback for when the client receives a response from the MQTT broker 

In [ ]:
def on_connect(client, userdata, flags, rc):
    print(f'Connected with result code {str(rc)}')
    client.subscribe('s337116')

## Define the callback for when a message is published on a subscribed topic

In [ ]:

def on_message(client, userdata, msg):
    message = msg.payload.decode()
    # parse json message
    message = json.loads(message)

    # get relevant data from message
    mac_address = message["mac_address"]
    timestamp = message["timestamp"]
    data = message["data"]

    # get redis client and timeseries names
    redis_client, temp_timeseries_name, humidity_timeseries_name = get_redis_client_instance(mac_address)

    # push each data row to the appropriate timeseries
    for d in data:
        name = d["name"]
        value = d["value"]
        timeseries = temp_timeseries_name if name == "temperature" else humidity_timeseries_name
        
        redis_client.ts().add(timeseries, timestamp, value)

    print(f"Received message '{message}' on topic {msg.topic}")

## Set the callbacks to the client object

In [ ]:
client.on_connect = on_connect
client.on_message = on_message

## Connect to the MQTT broker and wait for messages

In [ ]:
client.connect('broker.emqx.io', 1883)
client.loop_forever()

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=3880e510-b64c-4bb5-b488-c2122d5d9e2d' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>